# RQ4 — Does Week 3's format gap in J-Lens ranks mean a format gap in causal leverage?

This notebook **only reads saved results** (`results/rq4_raw_results.jsonl`, built from the per-evaluation checkpoints). It never loads a model and never runs inference. If the file is absent it says so and stops — nothing is simulated.

**Design (FROZEN_PROTOCOL.md v1).** Qwen3.5-0.8B, released `qwen-n1000` Jacobian Lens, ten Week 3 BBQ-Age base items × 4 conditions × {multiple choice, open ended}. A single-block, prefill-only edit at the `:` of `Answer:` at L8, L16, L22:
* **target** — clean projection ablation along `unit(J_lᵀ(mean U[stereotype ids] − mean U[counter ids]))` (a token-set contrast, *not* the Week 3 min-constituent rank statistic);
* **random** — three fixed-seed directions (11, 23, 47), **norm-matched displacement** to the target's measured ablation norm;
* **sham** — `banana` vs `table` through the same lens, norm-matched likewise.
Every displacement is capped at 10 % of the activation norm; target ablations above the cap are *safety-excluded*, not weakened.

**Estimands.** Each intervention is compared with its *own format's* clean baseline first. Primary continuous: |Δ teacher-forced log-prob margin| under target minus the mean under the three random seeds; then the paired MC − open difference, per layer, after averaging the four conditions within each base item. Co-primary behavioural: the same excess for *changed classified generated answers* (stereotype / counter / unknown-hedged / unparseable). Uncertainty: base-item clustered bootstrap (2 000 resamples, seed 20261020). With ten items, intervals are descriptive.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'rq4').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from rq4 import config as C, analysis as A, figures, store
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

n = store.combine(C.EVAL_DIR, C.RAW_JSONL, C.config_fingerprint()) if C.EVAL_DIR.exists() else 0
HAVE = n > 0
print(f'{n} saved evaluation rows (config {C.config_fingerprint()})' if HAVE else
      'NO SAVED RESULTS YET: Stage 1 has not been run. Run `python run_rq4.py --run-inference --i-confirm-no-heavy-experiment --stage 1`; nothing below is simulated.')

## 1. Run status and completion gate
Labels: `PILOT_ONLY` (fewer than ten base items), `INCONCLUSIVE` (completed but a gate item failed), `BLOCKED`. A gate pass is not evidence of an effect.

In [ ]:
if HAVE:
    df = A.load_results(C.RAW_JSONL)
    print(A.completion_status(df))
    display(df.groupby(['arm', 'status']).size().unstack(fill_value=0))

## 2. Displacement and controls
Target rows give the calibration; random and sham rows must have `calibration_ratio` ≈ 1 (bf16 rounding noise is expected) and `measured_ratio` ≤ 0.10. Safety-excluded and degenerate cells are listed rather than hidden.

In [ ]:
if HAVE:
    display(A.displacement_summary(df))
    display(df[df.status != 'ok'].groupby(['layer', 'status', 'status_detail'], dropna=False).size().rename('n'))

## 3. Raw generations and parser audit
Read these before trusting any category. `parse_rule` shows which rule fired; unparseable and hedged answers are kept as categories.

In [ ]:
if HAVE:
    display(A.category_rates(df).pivot_table(index=['prompt_format', 'layer', 'arm'], columns='category', values='rate', fill_value=0).round(3))
    display(df[(df.arm == 'clean')][['prompt_id', 'generation_raw', 'category', 'parse_rule', 'correct_role', 'follows_evidence']].head(16))

## 4. Within-format effects, then matched MC − open (per layer)
`excess_abs` = |Δ margin| (target) − mean |Δ margin| (3 random seeds); `excess_changed` is the same for the indicator "generated category differs from clean". Positive MC − open means the MC readout format shows the larger *excess* effect. Only pairs valid in **both** formats enter the difference.

In [ ]:
if HAVE:
    pairs = A.per_pair_effects(df)
    items = A.per_item(pairs)
    summ = A.summarize(items)
    display(summ.pivot(index='metric', columns='layer', values='mean').round(3))
    display(summ[summ.metric.isin(['diff_excess_abs', 'diff_excess_changed'])].round(3))

## 5. Individual base items and seed spread
The item table is the unit of inference (n = 10). The seed spread shows how much of the 'control' level is itself seed noise.

In [ ]:
if HAVE:
    display(items[['base_item_id', 'layer', 'n_conditions'] + [c for c in ['mc_excess_abs', 'oe_excess_abs', 'diff_excess_abs', 'mc_excess_changed', 'oe_excess_changed', 'diff_excess_changed'] if c in items]].round(3))
    display(A.seed_spread(pairs))

## 6. Figures (rebuilt from the saved results only)

In [ ]:
if HAVE:
    for p in figures.make_figures():
        display(Image(filename=str(p)))

## 7. How to read this (guard rails)
* A rank or teacher-forced-score shift **without** a change in generated answers is *not* a behavioural causal effect.
* A directional change that merely reverses which group is preferred is **not** neutralisation or debiasing.
* Ten base items, a development set already seen in Week 3: exploratory only. No held-out confirmation was run.
* A negative or null result (target ≤ controls, or no format difference) is a result; report it with the intervals above.
* Rank gaps use Week 3's min-constituent definition; the intervention direction uses the token-set contrast — different quantities.
* Fill `REPORT_TEMPLATE.md` from these tables; do not hand-edit numbers.